# Phase: is there temporal information in it?

Phase is circular: -pi and +pi are neighbours, so it is summarised by circular means and resultant lengths, never arithmetic averages. A component that holds a negligible share of the power has no reported phase at all. Overlapping windows advance their phases consistently by construction, so continuation is read between disjoint windows and against the nulls.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features.spectral` and `xauusd_quant.research` and read back from `results/spectral_research/<tf>/<series>/fft_<N>/`. Generate those first:

```
xq spectral-research --timeframe 5m --source regression_residual --fft-window 256
xq spectral-research --timeframe 5m --all-windows
xq spectral-research --all
```

Every number sits beside four null controls: **white noise**, a **detrended random walk** (the same regression and OU pipeline on a series with no structure), and **shuffled** and **block-bootstrapped** real returns. A spectral peak is not a cycle until it survives them.

No trading rule is defined or evaluated anywhere. No costs are applied. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.spectral_config import load_spectral_config

spectral = load_spectral_config(ROOT / "config" / "spectral.yaml")

TIMEFRAME = "5m"                    # change and re-run
SERIES = "regression_residual"      # regression_residual | ou_innovation | log_return
FFT_WINDOW = 256                    # N bars per window

results = spectral.results_dir(TIMEFRAME, SERIES, FFT_WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} {SERIES} N={FFT_WINDOW}. Run:  xq spectral-research "
        f"--timeframe {TIMEFRAME} --source {SERIES} --fft-window {FFT_WINDOW}")

def table(name):
    """Load a previously generated spectral table (None if it was not written)."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = results / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

payload = json.loads((results / "summary.json").read_text(encoding="utf-8"))
summary = payload["summary"]
lineage = payload["provenance"]["dataset"]
print(f"bars       : {summary['observations']:,} ({summary['valid_spectral_fraction']:.1%} "
      f"with a valid {FFT_WINDOW}-bar window)")
print(f"ticks      : {lineage['tick_dataset_version']} ({lineage['tick_dataset_status']})")
print(f"bars       : {lineage.get('bar_dataset_version')}")
if lineage.get("label"):
    print(f"WARNING    : {lineage['label']}")
for warning in payload.get("warnings", []):
    print(f"WARNING    : {warning}")

## Future change by phase sector of the dominant component

In [ ]:
table('phase_analysis')

In [ ]:
plot('phase_outcomes')

## Does the phase advance like a steady cycle between disjoint windows?

In [ ]:
table('phase_consistency')

## The dominant component carried forward against what happened

In [ ]:
table('phase_projection')